# Independent Cohort Acquisition — Calibration & Pilot Launcher (Colab GPU)

> **Launcher only.** Every step calls the production CLI `scripts/research/run_cohort_acquisition.py`;
> download, cropping, local remapping, inpainting, compositing, and QC are **not** re-implemented here.
> **Pinned code**: the repository is checked out *detached* at `EXPECTED_COMMIT` (Cell 2) and `HEAD` is asserted; a mutable branch head is never executed.

## What this notebook does
1. **Cell 1** defines fail-closed helpers (no side effects).
2. **Cell 2** mounts Google Drive (failure **stops**; there is no `/content` scratch fallback), checks out the pinned commit (a dirty clone **stops** and is left untouched), installs inference dependencies, runs fail-closed plan preflight, and runs `--check-models` (verifying model metadata, configs and weight access).
3. **Cell 3** executes the selected run (`EXECUTION_MODE = 'calibration'` for the 2-attempt approved calibration plan, `'diagnostic'` for the historical 6-attempt diagnostic run, or `'pilot'` for the 8-pair production pilot). The CLI checks the locked GPU policy (CUDA, >= 12 GiB VRAM; Colab T4/A100), and refuses an existing run directory unless `RESUME_RUN_ID` is set.
4. **Cell 4** runs only if Cell 3 completed in *this* session: it audits the run receipt against this run id, commit, and plan hashes, then writes a new `<RUN_ID>_package.zip` (never overwrites).

**Strict Separation of Execution Modes:**
- `calibration`: Exactly 2 attempts on candidate IND_COCO_SDXL_002 (EulerDiscreteScheduler, 30 steps, seed 20272319, guidance 7.5 vs 9.5, canvas 512×512, no crop). Fails closed if plan status != APPROVED. Uses sealed inputs from pilot-20261008T113700Z. Results do NOT enter official cohort.
- `diagnostic`: Historical 6 attempts (completed, 6/6 REJECT). Fails closed if plan status is PENDING.
- `pilot`: Production 8-pair pilot. Historical pilot `pilot-20261008T113700Z` is preserved intact.
- Full acquisition is blocked and cannot be launched from this notebook.


In [ ]:
# Cell 1: Fail-closed helpers (definitions only; nothing runs here)
import subprocess
import sys
from pathlib import Path


def run(cmd, cwd=None):
    """Run a command streaming stdout/stderr live; a non-zero exit raises CalledProcessError and stops the notebook."""
    cmd = [str(c) for c in cmd]
    print("$", " ".join(cmd), flush=True)
    with subprocess.Popen(
        cmd,
        cwd=cwd,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        encoding="utf-8",
        errors="replace",
        bufsize=1,
    ) as proc:
        if proc.stdout:
            for line in proc.stdout:
                sys.stdout.write(line)
                sys.stdout.flush()
        retcode = proc.wait()
        if retcode != 0:
            raise subprocess.CalledProcessError(retcode, cmd)


def _git(repo_dir, *args):
    return subprocess.run(["git", *args], cwd=repo_dir, check=True, capture_output=True, text=True).stdout.strip()


def checkout_pinned(repo_dir, repo_url, sha):
    """Detached checkout of exactly `sha`. A clone with local changes is never modified."""
    repo_dir = Path(repo_dir)
    if not (repo_dir / ".git").is_dir():
        run(["git", "clone", "--no-checkout", repo_url, repo_dir])
    else:
        dirty = _git(repo_dir, "status", "--porcelain")
        if dirty:
            raise RuntimeError(f"{repo_dir} has uncommitted changes; resolve them manually (nothing was modified):\n{dirty}")
    run(["git", "fetch", "origin", sha], cwd=repo_dir)
    run(["git", "checkout", "--detach", sha], cwd=repo_dir)
    head = _git(repo_dir, "rev-parse", "HEAD")
    if head != sha:
        raise RuntimeError(f"HEAD {head} != expected {sha}")
    return head


def mount_drive_or_fail(mount_fn, mount_point):
    """Mount Google Drive or stop; acquisition outputs are never written to scratch storage."""
    try:
        mount_fn(str(mount_point))
    except Exception as e:
        raise RuntimeError(f"Google Drive mount failed ({e}); stopping without creating any run.") from e
    my_drive = Path(mount_point) / "MyDrive"
    if not my_drive.is_dir():
        raise RuntimeError(f"Google Drive mount did not expose {my_drive}; stopping without creating any run.")
    return my_drive


In [ ]:
# Cell 2: Drive, pinned checkout, dependencies, plan preflight (any failure stops here)
import datetime
from google.colab import drive

EXPECTED_COMMIT = "38df28b4f7fcae0d8788057418410c27d1ca5852"
REPO_URL = "https://github.com/nomozer/forensics-web-lab.git"
REPO_DIR = Path("/content/forensics-web-lab")
RESUME_RUN_ID = None  # set to an existing run id to resume it (binding + SHA-256 are re-verified)

# Explicit execution mode selection: 'calibration' (2 attempts approved plan), 'diagnostic' (6 attempts), or 'pilot' (production pilot)
# Calibration and diagnostic require verified input PNGs on Drive from pilot-20261008T113700Z.
EXECUTION_MODE = "calibration"

if EXECUTION_MODE not in ("calibration", "diagnostic", "pilot"):
    raise RuntimeError(f"Missing or invalid EXECUTION_MODE ({EXECUTION_MODE!r}); must be 'calibration', 'diagnostic', or 'pilot'.")

MY_DRIVE = mount_drive_or_fail(drive.mount, "/content/drive")
RUNS_ROOT = MY_DRIVE / "Colab Notebooks" / "forensics-web-lab" / "independent_cohort_acquisition" / "runs"
EDIT_PLAN_PATH = REPO_DIR / "research/evidence/phase-4c.7b/content_grounded_pilot_plan.json"
DIAGNOSTIC_PLAN_PATH = REPO_DIR / "research/evidence/phase-4c.7b/content_grounded_diagnostic_plan.json"
CALIBRATION_PLAN_PATH = REPO_DIR / "research/evidence/phase-4c.7b/content_grounded_calibration_proposal.json"
DIAGNOSTIC_INPUTS_DIR = RUNS_ROOT / "pilot-20261008T113700Z"
CALIBRATION_INPUTS_DIR = RUNS_ROOT / "pilot-20261008T113700Z"
RUN_ID = RESUME_RUN_ID or ((("calib-" if EXECUTION_MODE == "calibration" else ("diag-" if EXECUTION_MODE == "diagnostic" else "pilot-"))) + datetime.datetime.now(datetime.timezone.utc).strftime("%Y%m%dT%H%M%SZ"))

checkout_pinned(REPO_DIR, REPO_URL, EXPECTED_COMMIT)
run([sys.executable, "-m", "pip", "install", "-q", "diffusers", "transformers", "accelerate", "safetensors", "huggingface_hub"])

if EXECUTION_MODE == "calibration":
    run([sys.executable, "scripts/research/run_cohort_acquisition.py", "--check-calibration-plan",
         "--calibration-plan-path", CALIBRATION_PLAN_PATH, "--calibration-inputs-dir", CALIBRATION_INPUTS_DIR], cwd=REPO_DIR)
elif EXECUTION_MODE == "diagnostic":
    run([sys.executable, "scripts/research/run_cohort_acquisition.py", "--check-diagnostic-plan",
         "--diagnostic-plan-path", DIAGNOSTIC_PLAN_PATH, "--diagnostic-inputs-dir", DIAGNOSTIC_INPUTS_DIR], cwd=REPO_DIR)
elif EXECUTION_MODE == "pilot":
    run([sys.executable, "scripts/research/run_cohort_acquisition.py", "--verify-plan"], cwd=REPO_DIR)

run([sys.executable, "scripts/research/run_cohort_acquisition.py", "--check-models"], cwd=REPO_DIR)
print(f"Preflight PASS | commit {EXPECTED_COMMIT} | mode {EXECUTION_MODE} | run {RUN_ID} -> {RUNS_ROOT / RUN_ID}")


In [ ]:
# Cell 3: Technical execution via the production CLI (calibration, diagnostic, or pilot)
globals().pop("PILOT_RUN_COMPLETED", None)
globals().pop("RUN_CONTEXT", None)

mode = globals().get("EXECUTION_MODE")
if mode not in ("calibration", "diagnostic", "pilot"):
    raise RuntimeError(
        f"Missing or invalid EXECUTION_MODE ({mode!r}); must be explicitly set to 'calibration', 'diagnostic', or 'pilot' before running."
    )

if mode == "calibration":
    calib_plan_path = globals().get("CALIBRATION_PLAN_PATH")
    calib_inputs_dir = globals().get("CALIBRATION_INPUTS_DIR")
    if not calib_plan_path or not calib_inputs_dir:
        raise RuntimeError("CALIBRATION_PLAN_PATH and CALIBRATION_INPUTS_DIR must be set for calibration mode.")
    cmd = [
        sys.executable, "scripts/research/run_cohort_acquisition.py",
        "--mode", "calibration", "--run-id", RUN_ID, "--expected-commit", EXPECTED_COMMIT,
        "--output-root", RUNS_ROOT, "--calibration-plan-path", calib_plan_path,
        "--calibration-inputs-dir", calib_inputs_dir,
    ]
elif mode == "diagnostic":
    diag_plan_path = globals().get("DIAGNOSTIC_PLAN_PATH")
    diag_inputs_dir = globals().get("DIAGNOSTIC_INPUTS_DIR")
    if not diag_plan_path or not diag_inputs_dir:
        raise RuntimeError("DIAGNOSTIC_PLAN_PATH and DIAGNOSTIC_INPUTS_DIR must be set for diagnostic mode.")
    cmd = [
        sys.executable, "scripts/research/run_cohort_acquisition.py",
        "--mode", "diagnostic", "--run-id", RUN_ID, "--expected-commit", EXPECTED_COMMIT,
        "--output-root", RUNS_ROOT, "--diagnostic-plan-path", diag_plan_path,
        "--diagnostic-inputs-dir", diag_inputs_dir,
    ]
elif mode == "pilot":
    cmd = [
        sys.executable, "scripts/research/run_cohort_acquisition.py",
        "--mode", "pilot", "--run-id", RUN_ID, "--expected-commit", EXPECTED_COMMIT,
        "--output-root", RUNS_ROOT, "--edit-plan-path", EDIT_PLAN_PATH, "--contact-sheet",
    ]

if globals().get("RESUME_RUN_ID"):
    cmd.append("--resume")

run(cmd, cwd=REPO_DIR)

completed_dir = RUNS_ROOT / RUN_ID
RUN_CONTEXT = {
    "run_dir": completed_dir,
    "run_id": RUN_ID,
    "mode": mode,
    "expected_commit": EXPECTED_COMMIT,
    "plan_path": (
        calib_plan_path if mode == "calibration"
        else (diag_plan_path if mode == "diagnostic" else EDIT_PLAN_PATH)
    ),
}
PILOT_RUN_COMPLETED = completed_dir


In [ ]:
# Cell 4: Audit THIS session's run, then package it (never a stale run or ZIP)
import shutil

context = globals().get("RUN_CONTEXT")
if not context or not isinstance(context, dict):
    raise RuntimeError("CLI run did not complete in this session (missing RUN_CONTEXT); nothing to audit or package.")

run_dir = context["run_dir"]
run_id = context["run_id"]
mode = context["mode"]
commit = context["expected_commit"]
plan_path = context["plan_path"]

if mode == "calibration":
    run([sys.executable, "scripts/research/run_cohort_acquisition.py", "--audit-run", run_dir,
         "--mode", "calibration", "--run-id", run_id, "--expected-commit", commit,
         "--calibration-plan-path", plan_path], cwd=REPO_DIR)
elif mode == "diagnostic":
    run([sys.executable, "scripts/research/run_cohort_acquisition.py", "--audit-run", run_dir,
         "--mode", "diagnostic", "--run-id", run_id, "--expected-commit", commit,
         "--diagnostic-plan-path", plan_path], cwd=REPO_DIR)
elif mode == "pilot":
    run([sys.executable, "scripts/research/run_cohort_acquisition.py", "--audit-run", run_dir,
         "--mode", "pilot", "--run-id", run_id, "--expected-commit", commit,
         "--edit-plan-path", plan_path], cwd=REPO_DIR)
else:
    raise RuntimeError(f"Unknown mode in RUN_CONTEXT: {mode!r}")

zip_base = RUNS_ROOT / f"{run_id}_package"
if Path(f"{zip_base}.zip").exists():
    raise RuntimeError(f"{zip_base}.zip already exists; refusing to overwrite.")
shutil.make_archive(str(zip_base), "zip", run_dir)
print(f"Audit PASS | package {zip_base}.zip | run dir: {run_dir}")
